# Treino — LateFusionSpatialCNN 11×11 com Center-Aware Pooling (SEM BRANCH MASKS)

Este notebook executa o treino e avaliação da **LateFusionCNN_CenterAware** removendo a branch de **MASKS**.

**Alteração em relação à versão standard:**
- As bandas de máscaras (`MASK`, `VALID`, `AVAIL`, `NODATA`) são automaticamente filtradas durante o carregamento de dados.
- A arquitetura cria apenas as branches dos sensores físicos (`S1_SAR`, `S2_OPTICAL`, `TOPOGRAPHY`, `ALOS2_SAR`, etc.) sem criar a branch `MASKS`.
- Todas as restantes componentes (Center-Aware Pooling, Weighted Loss, Stratified CV e Ensemble) mantêm-se rigorosamente idênticas.


In [9]:
# ============================================================
# 1. IMPORTS E CONFIGURAÇÃO GLOBAL
# ============================================================
import os
import time
import json
import re
import random
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import rasterio
import geopandas as gpd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Caminhos
PATCH_BASE_DIR = Path(r"D:\Tese_data_refinada\_example_patches_by_size")
VALIDATION_DIR = Path(r"D:\Tese_data_refinada\Areas_gpkg")

OUTPUT_BASE_DIR = Path("outputs_fch/spacial_cnn/late_fusion_cnn_11x11_center_aware_no_masks")
OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)

PLOTS_DIR = OUTPUT_BASE_DIR / "figuras"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Configurações
SEED = 42
TARGET_COLUMN = "rh_opt_098"
TARGET_SCALE = 100.0   # cm -> metros
PATCH_SIZE_TO_USE = 11
RAW_BASE_DIR = PATCH_BASE_DIR / f"patch_{PATCH_SIZE_TO_USE}x{PATCH_SIZE_TO_USE}"

N_SPLITS = 5
N_HEIGHT_BINS = 5
AUGMENTATION_MODE = "aug_on"

HIGH_HEIGHT_THRESHOLD_M = 20.0
HIGH_HEIGHT_WEIGHTS_TO_RUN = [1.5, 2.0]
ENSEMBLE_SEEDS = [42, 123, 2024]
FIXED_HEIGHT_BINS_M = [0, 5, 10, 15, 20, np.inf]

BATCH_SIZE = 64
LR = 3e-4
WEIGHT_DECAY = 0.0
EPOCHS = 70
PATIENCE = 15
LOG_EVERY = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)


Dispositivo: cuda
GPU: NVIDIA GeForce RTX 3060


In [10]:
# ============================================================
# 2. LEITURA DE DADOS E FILTRAGEM (SEM MASKS)
# ============================================================
def extract_point_id(filename):
    match = re.search(r"point_(\d+)", filename, re.IGNORECASE)
    if match:
        return int(match.group(1))
    nums = re.findall(r"\d+", filename)
    if nums:
        return int(nums[-1])
    raise ValueError(f"Não foi possível extrair point_id de: {filename}")

def build_target_dict(gdf, target_column):
    possible_id_cols = ["point_id", "POINT_ID", "id", "ID", "fid", "FID"]
    id_col = next((c for c in possible_id_cols if c in gdf.columns), None)
    ids = gdf[id_col].astype(int).values if id_col else gdf.index.astype(int).values
    targets = gdf[target_column].values
    return {int(pid): float(t) for pid, t in zip(ids, targets) if not pd.isna(t)}

def canonical_band_name(name):
    s = str(name)
    m = re.search(r"20\d{6}_.+", s)
    if m:
        s = m.group(0)
    s = re.sub(r"_(A|MOSAIC)$", "", s, flags=re.IGNORECASE)
    return s

def infer_sensor_group_from_name(name):
    u = str(name).upper()
    if any(k in u for k in ["SENTINEL1", "SENTINEL-1", "S1", "_VV", "_VH", "VV_", "VH_", "SAR"]):
        return "S1_SAR"
    s2_band_tokens = [f"B{i:02d}" for i in range(1, 13)] + ["B8A", "NDVI", "EVI", "NDWI", "NDMI", "NBR"]
    if any(k in u for k in ["SENTINEL2", "SENTINEL-2", "S2"] + s2_band_tokens):
        return "S2_OPTICAL"
    if any(k in u for k in ["DEM", "DTM", "DSM", "ELEV", "ELEVATION", "SLOPE", "ASPECT", "TPI", "TRI"]):
        return "TOPOGRAPHY"
    if any(k in u for k in ["MASK", "VALID", "AVAIL", "NODATA"]):
        return "MASKS"
    return "OTHER"

def load_target_canonical_bands():
    bands_csv = Path(r"C:\Users\gui\Documents\Tese\trabalho tese - Cópia\organizado\RFE\RFE_results\iter_27_bands.csv")
    if not bands_csv.exists():
        bands_csv = Path(r"C:\Users\gui\Documents\Tese\trabalho tese\organizado\RFE\RFE_results\iter_27_bands.csv")
    df = pd.read_csv(bands_csv)
    return set([canonical_band_name(name) for name in df['band_name']])

def get_valid_band_indices(metadata_path, exclude_masks=True):
    canonical_targets = load_target_canonical_bands()
    with open(metadata_path, "r", encoding="utf-8") as f:
        meta = json.load(f)
    raster_names = meta.get("raster_names", [])

    valid_indices = []
    for i, name in enumerate(raster_names):
        c_name = canonical_band_name(name)
        group = infer_sensor_group_from_name(name)
        # Se exclude_masks=True, exclui bandas do grupo MASKS
        if exclude_masks and group == "MASKS":
            continue
        if c_name in canonical_targets:
            valid_indices.append(i)

    return valid_indices, raster_names


In [11]:
# ============================================================
# 3. CARREGAMENTO COMPLETO DOS PATCHES (SEM MASKS)
# ============================================================
def load_all_regions_tifs(raw_base_dir, validation_dir, target_column="rh_opt_098", exclude_masks=True):
    raw_base_dir = Path(raw_base_dir)
    validation_dir = Path(validation_dir)

    X_list, y_list, region_list, pid_list = [], [], [], []
    channels_ref = None
    band_names_ref = None

    print(f"-> A ler TIFs e cruzar com GeoPackages (EXCLUDE_MASKS={exclude_masks})...")
    for area_dir in sorted(raw_base_dir.iterdir()):
        if not area_dir.is_dir(): continue
        area_name = area_dir.name
        gpkg_path = validation_dir / f"{area_name}_HEIGHT_Dataset.gpkg"
        if not gpkg_path.exists(): continue

        metas = list(area_dir.glob("*.json"))
        if not metas: continue

        gdf = gpd.read_file(gpkg_path)
        target_dict = build_target_dict(gdf, target_column)
        valid_idx, raster_names = get_valid_band_indices(metas[0], exclude_masks=exclude_masks)
        valid_band_names = [raster_names[i] for i in valid_idx]

        if channels_ref is None:
            channels_ref = len(valid_idx)
            band_names_ref = valid_band_names
            print(f"   [{area_name}] Número de canais ativos (sem masks): {channels_ref}")

        n_loaded = 0
        for tif in sorted(area_dir.rglob("*.tif")):
            pid = extract_point_id(tif.name)
            if pid not in target_dict: continue
            with rasterio.open(tif) as src:
                patch = src.read().astype(np.float32)[valid_idx]
            X_list.append(patch)
            y_list.append(target_dict[pid])
            region_list.append(area_name)
            pid_list.append(pid)
            n_loaded += 1

        print(f"   [{area_name}] Carregados: {n_loaded}")

    X = np.stack(X_list).astype(np.float32)
    y = np.array(y_list, dtype=np.float32)
    regions = np.array(region_list)
    point_ids = np.array(pid_list)

    print(f"\nResumo final (Sem Masks): X={X.shape} | y={y.shape}")
    return X, y, regions, point_ids, band_names_ref


In [12]:
# ============================================================
# 4. NORMALIZAÇÃO SEM LEAKAGE
# ============================================================
def normalize_from_train(X_train_raw, X_val_raw):
    mean = np.nanmean(X_train_raw, axis=(0, 2, 3), keepdims=True)
    std  = np.nanstd(X_train_raw,  axis=(0, 2, 3), keepdims=True)
    std  = np.where(std == 0, 1e-6, std)

    X_train = np.nan_to_num((X_train_raw - mean) / std, nan=0., posinf=0., neginf=0.).astype(np.float32)
    X_val   = np.nan_to_num((X_val_raw   - mean) / std, nan=0., posinf=0., neginf=0.).astype(np.float32)
    return X_train, X_val, mean, std


In [13]:
# ============================================================
# 5. ARQUITETURA LATE FUSION SPATIAL CNN (SEM BRANCH MASKS)
# ============================================================
from collections import OrderedDict

def infer_late_fusion_channel_groups(band_names):
    groups = OrderedDict()
    for idx, name in enumerate(band_names):
        group = infer_sensor_group_from_name(name)
        if group == "MASKS":
            continue  # Garante que a branch MASKS é ignorada se existir alguma banda
        groups.setdefault(group, []).append(idx)

    preferred_order = ["S1_SAR", "S2_OPTICAL", "TOPOGRAPHY", "OTHER"]
    ordered = OrderedDict()
    for group in preferred_order:
        if group in groups:
            ordered[group] = groups[group]
    for group in groups:
        if group not in ordered:
            ordered[group] = groups[group]

    print("\nGrupos de canais ativos em Late Fusion (SEM MASKS):")
    for group, idxs in ordered.items():
        print(f"  {group}: {len(idxs)} canais")
    return ordered

class SensorSpatialBranchCenterAware(nn.Module):
    def __init__(self, in_channels: int, branch_channels: int = 32, center_radius: int = 1, global_pool_size: int = 2):
        super().__init__()
        self.center_radius = int(center_radius)
        self.global_pool_size = int(global_pool_size)

        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, branch_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(branch_channels),
            nn.GELU(),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(branch_channels, branch_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(branch_channels),
            nn.GELU(),
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(branch_channels, branch_channels, kernel_size=3, padding=2, dilation=2, bias=False),
            nn.BatchNorm2d(branch_channels),
            nn.GELU(),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((global_pool_size, global_pool_size))
        self.flatten = nn.Flatten()
        self.out_features = branch_channels * global_pool_size * global_pool_size + branch_channels

    def extract_center_features(self, x: torch.Tensor) -> torch.Tensor:
        _, _, h, w = x.shape
        cy, cx = h // 2, w // 2
        y0 = max(cy - self.center_radius, 0)
        y1 = min(cy + self.center_radius + 1, h)
        x0 = max(cx - self.center_radius, 0)
        x1 = min(cx + self.center_radius + 1, w)
        return x[:, :, y0:y1, x0:x1].mean(dim=(2, 3))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        global_features = self.flatten(self.global_pool(x))
        center_features = self.extract_center_features(x)
        return torch.cat([global_features, center_features], dim=1)

class LateFusionSpatialCNN_CenterAware_NoMasks(nn.Module):
    """
    LateFusionSpatialCNN Center-Aware sem branch MASKS.
    """
    def __init__(self, in_channels: int, channel_groups, branch_channels: int = 32, center_radius: int = 1, global_pool_size: int = 2):
        super().__init__()
        self.group_names = list(channel_groups.keys())
        self.group_indices = [list(channel_groups[g]) for g in self.group_names]

        self.branches = nn.ModuleDict()
        total_features = 0
        for group_name, idxs in zip(self.group_names, self.group_indices):
            module_key = re.sub(r"[^A-Za-z0-9_]", "_", group_name)
            branch = SensorSpatialBranchCenterAware(
                in_channels=len(idxs),
                branch_channels=branch_channels,
                center_radius=center_radius,
                global_pool_size=global_pool_size,
            )
            self.branches[module_key] = branch
            total_features += branch.out_features

        self.module_keys = [re.sub(r"[^A-Za-z0-9_]", "_", g) for g in self.group_names]

        self.regressor = nn.Sequential(
            nn.Linear(total_features, 128, bias=False),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = []
        for module_key, indices in zip(self.module_keys, self.group_indices):
            features.append(self.branches[module_key](x[:, indices, :, :]))
        return self.regressor(torch.cat(features, dim=1)).squeeze(1)


In [14]:
# ============================================================
# 6. MÉTRICAS E DATASET DATA LOADER
# ============================================================
class PatchDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.augment = augment

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x_item = self.X[idx]
        y_item = self.y[idx]
        if self.augment:
            k = random.randint(0, 3)
            if k > 0:
                x_item = torch.rot90(x_item, k, dims=[1, 2])
            if random.random() > 0.5:
                x_item = torch.flip(x_item, dims=[1])
            if random.random() > 0.5:
                x_item = torch.flip(x_item, dims=[2])
        return x_item, y_item

def regression_metrics(y_true, y_pred):
    y_true_np = np.array(y_true).flatten()
    y_pred_np = np.array(y_pred).flatten()
    rmse = float(np.sqrt(mean_squared_error(y_true_np, y_pred_np)))
    mae = float(mean_absolute_error(y_true_np, y_pred_np))
    bias = float(np.mean(y_pred_np - y_true_np))
    r2 = float(r2_score(y_true_np, y_pred_np)) if len(y_true_np) > 1 else float("nan")
    pct_rmse = (rmse / np.mean(y_true_np)) * 100 if np.mean(y_true_np) > 0 else float("nan")
    return {"n": len(y_true_np), "RMSE": rmse, "%RMSE": pct_rmse, "MAE": mae, "R2": r2, "Bias": bias}

class WeightedMSELoss(nn.Module):
    def __init__(self, threshold_m=20.0, weight_high=1.5):
        super().__init__()
        self.threshold_m = float(threshold_m)
        self.weight_high = float(weight_high)

    def forward(self, y_pred, y_true):
        diff_sq = (y_pred - y_true) ** 2
        weights = torch.where(y_true >= self.threshold_m, self.weight_high, 1.0)
        return torch.mean(weights * diff_sq)


In [15]:
# ============================================================
# 7. FUNÇÃO DE TREINO POR FOLD
# ============================================================
def train_model(
    model, train_loader, val_loader, model_name, loss_fn,
    epochs=EPOCHS, patience=PATIENCE, lr=LR, log_every=LOG_EVERY
):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)
    ckpt_path = OUTPUT_BASE_DIR / f"ckpt_{model_name}.pth"

    best_val_loss = np.inf
    epochs_no_impr = 0

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []
        for bX, by in train_loader:
            bX, by = bX.to(device), by.to(device)
            optimizer.zero_grad()
            pred = model(bX)
            loss = loss_fn(pred, by)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for bX, by in val_loader:
                val_preds.append(model(bX.to(device)).cpu().numpy())
                val_targets.append(by.numpy())

        val_targets = np.concatenate(val_targets)
        val_preds = np.concatenate(val_preds)
        val_mse = float(mean_squared_error(val_targets, val_preds))
        scheduler.step(val_mse)

        if val_mse < best_val_loss:
            best_val_loss = val_mse
            epochs_no_impr = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            epochs_no_impr += 1
            if epochs_no_impr >= patience:
                break

    model.load_state_dict(torch.load(ckpt_path))
    return model


In [16]:
# ============================================================
# 8. PIPELINE DE EXECUÇÃO CROSS-VALIDATION + ENSEMBLE
# ============================================================
print("=== A Iniciar Treino da LateFusionCNN (SEM MASKS) ===")
raw_base_dir = RAW_BASE_DIR
X_raw, y_raw, regions, point_ids, band_names_ref = load_all_regions_tifs(
    raw_base_dir, VALIDATION_DIR, TARGET_COLUMN, exclude_masks=True
)

y_m = y_raw / TARGET_SCALE
channel_groups = infer_late_fusion_channel_groups(band_names_ref)

print(f"\nCanais totais de entrada para o modelo: {X_raw.shape[1]}")
for g, idxs in channel_groups.items():
    print(f"  Ramo {g}: {len(idxs)} canais")

# Stratified K-Fold por classes de altura
height_bins = pd.qcut(y_m, q=N_HEIGHT_BINS, labels=False, duplicates='drop')
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

ensemble_val_preds = np.zeros((len(ENSEMBLE_SEEDS), len(y_m)), dtype=np.float32)

for seed_idx, seed in enumerate(ENSEMBLE_SEEDS):
    print(f"\n--- ENSEMBLE SEED {seed} ({seed_idx + 1}/{len(ENSEMBLE_SEEDS)}) ---")
    set_seed(seed)
    fold_preds = np.zeros(len(y_m), dtype=np.float32)

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_raw, height_bins), 1):
        X_tr_raw, y_tr = X_raw[train_idx], y_m[train_idx]
        X_va_raw, y_va = X_raw[val_idx], y_m[val_idx]

        X_tr, X_va, _, _ = normalize_from_train(X_tr_raw, X_va_raw)
        train_ds = PatchDataset(X_tr, y_tr, augment=True)
        val_ds = PatchDataset(X_va, y_va, augment=False)

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

        model = LateFusionSpatialCNN_CenterAware_NoMasks(in_channels=X_tr.shape[1], channel_groups=channel_groups)
        loss_fn = WeightedMSELoss(threshold_m=HIGH_HEIGHT_THRESHOLD_M, weight_high=1.5)

        model = train_model(model, train_loader, val_loader, f"seed_{seed}_fold_{fold}", loss_fn)

        model.eval()
        with torch.no_grad():
            preds = []
            for bX, _ in val_loader:
                preds.append(model(bX.to(device)).cpu().numpy())
            fold_preds[val_idx] = np.concatenate(preds)

        fold_metrics = regression_metrics(y_va, fold_preds[val_idx])
        print(f"  Fold {fold} | RMSE: {fold_metrics['RMSE']:.3f}m | %RMSE: {fold_metrics['%RMSE']:.2f}% | R²: {fold_metrics['R2']:.3f}")

    ensemble_val_preds[seed_idx] = fold_preds
    seed_metrics = regression_metrics(y_m, fold_preds)
    print(f"-> Resultado acumulado Seed {seed}: RMSE: {seed_metrics['RMSE']:.3f}m | %RMSE: {seed_metrics['%RMSE']:.2f}% | R²: {seed_metrics['R2']:.3f}")

# Média do Ensemble
final_ensemble_preds = np.mean(ensemble_val_preds, axis=0)
final_metrics = regression_metrics(y_m, final_ensemble_preds)

print("\n==================================================")
print("=== RESULTADO FINAL ENSEMBLE (SEM MASKS) ===")
print(f"  N Amostras: {final_metrics['n']}")
print(f"  RMSE:       {final_metrics['RMSE']:.4f} m")
print(f"  %RMSE:      {final_metrics['%RMSE']:.2f} %")
print(f"  MAE:        {final_metrics['MAE']:.4f} m")
print(f"  R²:         {final_metrics['R2']:.4f}")
print(f"  Bias:       {final_metrics['Bias']:.4f} m")
print("==================================================")

# Guardar resultados
df_res = pd.DataFrame([final_metrics])
df_res.to_csv(OUTPUT_BASE_DIR / "metrics_ensemble_no_masks.csv", index=False)
print(f"Métricas salvas em: {OUTPUT_BASE_DIR / 'metrics_ensemble_no_masks.csv'}")


=== A Iniciar Treino da LateFusionCNN (SEM MASKS) ===
-> A ler TIFs e cruzar com GeoPackages (EXCLUDE_MASKS=True)...
   [Mafra] Número de canais ativos (sem masks): 42
   [Mafra] Carregados: 414
   [Monsanto] Carregados: 423
   [Pombal] Carregados: 800
   [SerraLousa] Carregados: 5376
   [SintraCascais] Carregados: 636
   [VPA] Carregados: 515

Resumo final (Sem Masks): X=(8164, 42, 11, 11) | y=(8164,)

Grupos de canais ativos em Late Fusion (SEM MASKS):
  S1_SAR: 19 canais
  S2_OPTICAL: 23 canais

Canais totais de entrada para o modelo: 42
  Ramo S1_SAR: 19 canais
  Ramo S2_OPTICAL: 23 canais

--- ENSEMBLE SEED 42 (1/3) ---
  Fold 1 | RMSE: 5.603m | %RMSE: 35.90% | R²: 0.473
  Fold 2 | RMSE: 5.448m | %RMSE: 35.00% | R²: 0.496
  Fold 3 | RMSE: 5.448m | %RMSE: 34.69% | R²: 0.511
  Fold 4 | RMSE: 5.481m | %RMSE: 34.96% | R²: 0.517
  Fold 5 | RMSE: 5.511m | %RMSE: 35.63% | R²: 0.479
-> Resultado acumulado Seed 42: RMSE: 5.499m | %RMSE: 35.23% | R²: 0.496

--- ENSEMBLE SEED 123 (2/3) ---
 